In [1]:
import os
from datetime import date
from typing import Optional, Union

import matplotlib.pyplot as plt
import numpy as np
import porepy as pp

from src.tpf_lab.models.run_models import run_time_dependent_model
from src.tpf_lab.models.two_phase_flow_model import TwoPhaseFlow
from src.tpf_lab.utils import rm_out_padding

rm_out_padding()

### Geometrical setup
All the following tests run on the the square domain
$\Omega=\{(x,y)\mid 0\leq x,y\leq2\},$ which is disected into $20\times20$ grid cells of
equal size.

If not specified otherwise, homogeneous Neumann boundary conditions are applied to the
western, northern and eastern side of the domain, while homogeneous Dirichlet boundary
conditions are applied to the southern side of the domain.

### Linear capillary pressure model

To test that our model works, we let it run with a linear cap. pressure model first:
$$p_c=c\cdot S_w.$$
In order for the setup to pose at least some challenge, we introduce a wetting source in
in one of the four center cells of the domain, i.e.
$$
f_{w,i}=\begin{cases}
0.2,\quad&\text{if }i=209,\\
0,&\text{otherwise},
\end{cases}
$$
where $V_{209}=\{(x,y)\mid 1\leq x\leq1.1, 0.9\leq y\leq1\}.$

In [2]:
w_source_cell_index = 209
source = 0.5

class TwoPhaseFlow_WSource(TwoPhaseFlow):
    def _source_w(self, g: pp.Grid) -> np.ndarray:
        array: np.ndarray = super()._source_w(g)
        array[w_source_cell_index] = source
        return array

In [ ]:
pcap_model = "linear"

model = TwoPhaseFlow_WSource(
    {
    "formulation": "n_pressure_w_saturation",
    "file_name": f"pcap_{pcap_model}_source_{source}",
    "folder_name": os.path.join(
        "results",
        "setup_tests",
        f"{date.today().strftime('%Y-%m-%d')}_pcap_{pcap_model}"
        ),
    }
)
model._grid_size: int = 20
model._phys_size: int = 2
model._cap_pressure_model = pcap_model
model._time_step = 0.1
model._schedule: np.ndarray = np.array([0, 20])

run_time_dependent_model(model, {"nl_convergence_tol": 1e-10})

#### No cap. pressure model
We also run with a constant cap. pressure of $p_c=0.$

In [ ]:
pcap_model = None

model = TwoPhaseFlow_WSource(
    {
    "formulation": "n_pressure_w_saturation",
    "file_name": f"pcap_{pcap_model}_source_{source}",
    "folder_name": os.path.join(
        "results",
        "setup_tests",
        f"{date.today().strftime('%Y-%m-%d')}_pcap_{pcap_model}"
        ),
    }
)
model._grid_size: int = 20
model._phys_size: int = 2
model._cap_pressure_model = pcap_model
model._time_step = 0.1
model._schedule: np.ndarray = np.array([0, 20])

run_time_dependent_model(model, {"nl_convergence_tol": 1e-10})

#### Van Genuchten cap. pressure model
Next we try the *van Genuchten* model:
$$
\hat{S}_w=(1+(\beta_gp_c)^{n_g})^{-m_g}\text{ for the normalized saturation }\hat{S}_w=\frac{S_w-S_w^{min}}{S_w^{max}-S_w^{min}}\\
\implies p_c(\hat{S}_w)=\frac{(\hat{S}_w^{m_g}-1)^{-n_g}}{\beta_g}.
$$

In [ ]:
pcap_model = "van Genuchten"

model = TwoPhaseFlow_WSource(
    {
    "formulation": "n_pressure_w_saturation",
    "file_name": f"pcap_{pcap_model}_source_{source}",
    "folder_name": os.path.join(
        "results",
        "setup_tests",
        f"{date.today().strftime('%Y-%m-%d')}_pcap_{pcap_model}"
        ),
    }
)
model._grid_size: int = 20
model._phys_size: int = 2
model._cap_pressure_model = pcap_model
model._time_step = 0.1
model._schedule: np.ndarray = np.array([0, 20])

run_time_dependent_model(model, {"nl_convergence_tol": 1e-10})

#### Brooks-Corey cap. pressure model
And the *Brooks-Corey* model:
$$
\begin{align*}
    \hat{S}_w=
    \begin{cases}
        (p_c/p_e)^{-n_b},\quad&\text{if }p_c>p_e,\\
        1,&\text{if }p_c\leq p_e
    \end{cases}\\
    \implies p_c(\hat{S}_w)=
    \begin{cases}
        \hat{S_w}^{n_b}p_e,\quad&\text{if ???}\\
        ???,
    \end{cases}
\end{align*}
$$
**Note,** that with the default value $n_b=1,$ this is little more than a linear model
scaled with entry pressure.

In [ ]:
pcap_model = "Brooks-Corey"

model = TwoPhaseFlow_WSource(
    {
    "formulation": "n_pressure_w_saturation",
    "file_name": f"pcap_{pcap_model}_source_{source}",
    "folder_name": os.path.join(
        "results",
        "setup_tests",
        f"{date.today().strftime('%Y-%m-%d')}_pcap_{pcap_model}"
        ),
    }
)
model._grid_size: int = 20
model._phys_size: int = 2
model._cap_pressure_model = pcap_model
model._time_step = 0.1
model._schedule: np.ndarray = np.array([0, 20])

run_time_dependent_model(model, {"nl_convergence_tol": 1e-10})

#### Wetting pressure - wetting saturation formulation
For comparison!

In [ ]:
model = TwoPhaseFlow_WSource(
    {
    "formulation": "n_pressure_w_saturation",
    "file_name": f"{date.today().strftime('%Y-%m-%d')}",
    "folder_name": os.path.join(
        "results",
        "setup_tests",
        "p_w_S_w_formulation",
        f"{date.today().strftime('%Y-%m-%d')}_pcap_Brooks-Corey"
        ),
    }
)
model._grid_size: int = 20
model._phys_size: int = 2
model._cap_pressure_model = "Brooks-Corey"
model._schedule: np.ndarray = np.array([0, 50])
model._time_step = 0.2

run_time_dependent_model(model, {"max_iterations": 30, "nl_convergence_tol": 1e-5})